# 1. Essential matrix definition

The essential matrix $E$ is a $3\times3$ matrix that describes the geometric relationship between two views of the same scene, captured by calibrated cameras.

Its primary purpose is to help us recover the relative rotation and translation direction between two camera positions using corresponding image points.
In Visual Odometry (VO) and Visual SLAM, this allows us to estimate how a camera moved between two frames.

Essential matrix definition:
$$E=[t]_{\times}R$$

Where:
- $[t]_{\times}=\begin{bmatrix}
        0 & -t_z & t_y \\
        t_z & 0 & -t_x \\
        -t_y & t_x & 0
    \end{bmatrix}$ – the skew-symmetric matrix corresponding to the translation vector.

- $R$ - 3x3 rotation matrix


## Essential matrix usage

$E=[t]_{\times}R$

$P_2 = RP_1 + t$ – coordinate transformation from camera 1 to camera 2 of real-world point $P$.

# 2. Essential matrix computation. Five points method.

Transform each pixel point $p$ into a normalized camera ray:
$$x=K^{-1}p$$

The fundamental equation is:

$$x_2^TEx_1=0$$

Equation is based on the fact that three vectors are coplanar if their scalar triple product is zero. Vectors $x_1, x_2, t$ should be coplanar (epipolar geometry triangle).

**Note**: it does not tell us the 3D coordinates of the point. It only restricts where a corresponding point can appear in the second image.

Matrix $E$ has 5 degrees of freedom (3 for rotation and 2 for translation, because translation magnitude is unknown).
We cannot solve five linear equations for $E$. Because we also need to enforce the singular values to have the form $(\sigma, \sigma, 0)$.
The five-point method must enforce that structure, which makes the problem nonlinear (need to solve the polynom of 10th degree).

```python
E, mask = cv2.findEssentialMat(
    pts1,
    pts2,
    K,
    method=cv2.RANSAC
)
```

Note: RANSAC is used to improve the robustness of the estimation, but we need more than 5 points for this method.

# 3. Decompose Essential matrix:

1. First, apply SVD decomposition.

    $E=U\Sigma V^T$

    For an ideal Essential Matrix, the singular values have the special structure:

    $\Sigma=\begin{bmatrix}
        \sigma & 0 & 0 \\
        0 & \sigma & 0 \\
        0 & 0 & 0
    \end{bmatrix}$

   Because $E$ itself has an arbitrary scale, we normalize this to:

   $\Sigma=\begin{bmatrix}
        1 & 0 & 0 \\
        0 & 1 & 0 \\
        0 & 0 & 0
    \end{bmatrix}$
2. Then, extract target components
     $t=\pm u_3$ – the third column of $U$, translation candidates.

     Construct a special matrix:

     $W=\begin{bmatrix}
             0 & -1 & 0 \\
             1 & 0 & 0 \\
             0 & 0 & 1
         \end{bmatrix}$

     Extract $R$ candidates:

     $R_1=UWV^T$

     $R_2=UW^TV^T$

    ```python
    _, R, t, mask = cv2.recoverPose(
        E,
        pts1,
        pts2,
        K
    )
    ```

# 4. Important mathematical properties

The essential matrix has two particularly important properties:
$$
\operatorname{rank}(\mathbf E)=2
$$

And its singular values must satisfy:
$$
\sigma_1=\sigma_2>0,\qquad \sigma_3=0
$$

These are not arbitrary requirements. They follow directly from its structure $\mathbf E=[\mathbf t]_\times\mathbf R$.

The skew-symmetric matrix $[\mathbf t]_\times$ has rank 2 for nonzero translation because:
$$
[\mathbf t]_\times\mathbf t=0
$$

This means that $\mathbf t$ belongs to its null space.

Since $\mathbf R$ is an invertible rotation matrix, multiplying by it preserves the rank.

The two equal nonzero singular values are another consequence of the cross-product matrix structure and the fact that rotations preserve singular values.

These properties are used to validate and correct estimated essential matrices.

# 5. Relation with Fundamental matrix

$$
\boxed{\mathbf E=\mathbf K_2^T\mathbf F\mathbf K_1}
$$

# 6. Why is the essential matrix equal to $\mathbf E=[\mathbf t]_\times\mathbf R$?

We define the transformation from camera 1 coordinates to camera 2 coordinates as:
$$
\mathbf P_2=\mathbf R\mathbf P_1+\mathbf t
$$

Here, $\mathbf t$ is the position of camera 1's origin expressed in camera 2's coordinate system. It is not directly the displacement of camera 2 expressed in camera 1's system.

Because we only know the image projections, we can express the 3D points as:
$$
\mathbf P_1=\lambda_1\mathbf x_1
$$

$$
\mathbf P_2=\lambda_2\mathbf x_2
$$

where $\lambda_1,\lambda_2$ represent unknown depths.

Substituting:
$$
\lambda_2\mathbf x_2
=
\lambda_1\mathbf R\mathbf x_1+\mathbf t
$$

Now take the cross product of both sides with $\mathbf t$:
$$
\lambda_2(\mathbf t\times\mathbf x_2)
=
\lambda_1(\mathbf t\times\mathbf R\mathbf x_1)
$$

The term $\mathbf t\times\mathbf t$ disappears because it equals zero.

Next, take the dot product with $\mathbf x_2$. Since a cross product is perpendicular to its operands:
$$
\mathbf x_2^T(\mathbf t\times\mathbf x_2)=0
$$

Therefore:
$$
\mathbf x_2^T(\mathbf t\times\mathbf R\mathbf x_1)=0
$$

We can represent the cross product using a skew-symmetric matrix:
$$
[\mathbf t]_\times=
\begin{bmatrix}
0&-t_z&t_y\\
t_z&0&-t_x\\
-t_y&t_x&0
\end{bmatrix}
$$

This gives:
$$
\mathbf x_2^T[\mathbf t]_\times\mathbf R\mathbf x_1=0
$$

And finally:
$$
\boxed{\mathbf E=[\mathbf t]_\times\mathbf R}
$$

This derivation shows why the essential matrix contains information about both rotation and translation.